# Kaggle shared research pipeline
Smoke checks only. Pin the reviewed commit. Research runs require approval and verified durable storage.
Python 3.10/3.11 with torch 2.5.1 is the locked profile. A newer runtime must use an isolated compatible environment or be explicitly qualified; do not silently change package versions.

In [ ]:
from pathlib import Path
import subprocess, sys, uuid
if sys.version_info[:2] not in {(3,10),(3,11)}:
    raise RuntimeError("Select/provision Python 3.10 or 3.11 for the pinned research profile; see docs/platforms.md")
ROOT = Path.cwd()  # Open from the pinned repository checkout
subprocess.run([sys.executable,"-m","pip","install","-r",str(ROOT/"requirements.txt")],check=True)
subprocess.run([sys.executable,"-m","pip","install","-e",str(ROOT),"--no-deps"],check=True)


In [ ]:
DATA = ROOT / "data/processed/tiny-shakespeare"
if not DATA.exists():
    subprocess.run([sys.executable,"-m","nano_lm.prepare","--output",str(DATA)],check=True)
OUT = ROOT / "outputs/logs" / ("gpu_audit_" + uuid.uuid4().hex)
subprocess.run([sys.executable,"-m","nano_lm.verification","--root",str(ROOT),"--dataset",str(DATA),"--output",str(OUT),"--device","cuda","--precision","float16"],check=True)


Before releasing the runtime: export snapshots to a mounted persistent destination with `nano_lm.storage export`, verify their manifest, download/save notebook output, and restore/test from the copy. `/content` and `/kaggle/working` are ephemeral until saved. See docs/storage.md.